# 05 — Iterative State Analysis

Per §5.6.5, cumulative recall is computed over the union of raw retrieval sets through iteration t; marginal recall gain is the difference between adjacent cumulative values; NewRate measures newly seen documents in the current retrieval set; overlap is the Jaccard similarity between successive raw retrieval sets. This notebook does not treat MIGRES filtering as part of Recall@K itself.

In [ ]:
from pathlib import Path
import sys, json, yaml, platform
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print("Project root:", ROOT)
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Selected device:", DEVICE)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    DEVICE = "cpu"
    print("Torch/GPU probe unavailable:", type(e).__name__, e)
PLAN = yaml.safe_load((ROOT / "configs" / "experiments.yaml").read_text())
BASE_CONFIG = [ROOT / "configs" / "default.yaml", ROOT / "configs" / "gemini.yaml"]


In [ ]:
RUN_DIRS_BY_DATASET={}  # {'hotpotqa': {'sar': Path(...)}, 'musique': {...}, 'natural_questions': {...}}
LIMIT=None


In [ ]:
from sar_migres.utils.config import load_config
from sar_migres.corpus.models import Corpus
from sar_migres.datasets.loader import load_phase_questions,dataset_statistics
from sar_migres.experiments.iterative import iterative_metrics_from_run
base=load_config(*BASE_CONFIG)
if not RUN_DIRS_BY_DATASET:
    print('Preflight only. Set RUN_DIRS_BY_DATASET to completed run directories for each of the three datasets before loading data.')
else:
    corpus=Corpus.from_file(base.data.corpus_path,lazy=True,embed_template=base.embedding.embed_template,max_malformed_fraction=base.data.max_malformed_fraction,max_documents=base.data.max_documents)
    for DATASET in base.data.datasets:
        RUN_DIRS=RUN_DIRS_BY_DATASET.get(DATASET, {})
        items=load_phase_questions(DATASET,base.data.dataset_paths[DATASET],'final',limit=LIMIT,final_split=base.data.final_split,validation_fraction=base.data.validation_fraction,validation_seed=base.data.validation_seed)
        print('\n=== Dataset:',DATASET,'===')
        print('Dataset statistics:',dataset_statistics(items))
        for name,path in RUN_DIRS.items():
            print('\n###',name)
            print(json.dumps(iterative_metrics_from_run(path,items,corpus),indent=2,allow_nan=True))
